# 07 decoding 策略和生成参数

目标：实践 greedy、beam search、temperature、top-k、top-p、repetition penalty 等生成参数，并把它们和确定性、创造性、重复、延迟联系起来。


## 运行环境准备

这个 notebook 会真实加载模型并生成多次。CPU 也能运行，只是速度会慢一些。


In [ ]:
from pathlib import Path

requirements_path = Path("requirements.txt")
if not requirements_path.exists():
    requirements_path = Path("../requirements.txt")

%pip install -r {requirements_path}


In [ ]:
import os
import random
from pathlib import Path

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = os.getenv("MODEL_ID", "Qwen/Qwen2.5-0.5B-Instruct")
MODEL_SOURCE = os.getenv("MODEL_SOURCE", "modelscope").lower()


def resolve_model_path(model_id):
    if Path(model_id).exists():
        return model_id
    if MODEL_SOURCE != "modelscope":
        return model_id

    from modelscope import snapshot_download
    return snapshot_download(model_id)


def seed_everything(seed=42):
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


MODEL_PATH = resolve_model_path(MODEL_ID)
print("MODEL_ID =", MODEL_ID)
print("MODEL_PATH =", MODEL_PATH)


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16
if not torch.cuda.is_available():
    dtype = torch.float32

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    torch_dtype=dtype,
    device_map="auto" if torch.cuda.is_available() else None,
    trust_remote_code=True,
)
model.eval()
print("model dtype:", dtype)
print("first parameter device:", next(model.parameters()).device)


## 1. 统一生成函数

为了公平比较，后面的实验使用同一段 prompt，只替换生成参数。


In [ ]:
messages = [
    {"role": "system", "content": "你是一个讲解清楚但不啰嗦的机器学习面试官。"},
    {"role": "user", "content": "请用要点解释大模型部署中为什么需要 KV cache，并给一个风险点。"},
]

prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)
inputs = tokenizer(prompt, return_tensors="pt")
device = next(model.parameters()).device
inputs = {key: value.to(device) for key, value in inputs.items()}


def generate_once(name, seed=42, **generate_kwargs):
    seed_everything(seed)
    defaults = dict(
        max_new_tokens=100,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    defaults.update(generate_kwargs)

    with torch.inference_mode():
        output_ids = model.generate(**inputs, **defaults)

    new_tokens = output_ids[0, inputs["input_ids"].shape[1]:]
    text = tokenizer.decode(new_tokens, skip_special_tokens=True)
    print("=" * 80)
    print(name)
    print("new token count:", int(new_tokens.numel()))
    print(text)


## 2. greedy、beam search 和 sampling

- greedy 每步选概率最高的 token，稳定但可能死板。
- beam search 保留多个候选路径，适合翻译/摘要这类有明确目标的任务，开放问答中可能更模式化。
- sampling 从概率分布采样，适合开放生成，但要控制随机性。


In [ ]:
generate_once("greedy", do_sample=False)
generate_once("beam search", do_sample=False, num_beams=3, early_stopping=True)
generate_once("sampling", do_sample=True, temperature=0.8, top_p=0.9)


## 3. temperature 控制分布尖锐程度

温度越低越保守，温度越高越发散。生产里通常给不同任务设置不同默认值，并限制用户可调范围。


In [ ]:
for temperature in [0.2, 0.7, 1.2]:
    generate_once(
        f"temperature={temperature}",
        do_sample=True,
        temperature=temperature,
        top_p=0.9,
        seed=7,
    )


## 4. top-k、top-p 和 repetition penalty

`top_k` 保留概率最高的 k 个 token；`top_p` 保留累计概率达到 p 的候选集合；`repetition_penalty` 用来抑制重复，但过大可能伤害正常术语复用。


In [ ]:
settings = [
    ("top_k=10", dict(do_sample=True, temperature=0.8, top_k=10)),
    ("top_p=0.6", dict(do_sample=True, temperature=0.8, top_p=0.6)),
    ("top_p=0.95", dict(do_sample=True, temperature=0.8, top_p=0.95)),
    ("repetition_penalty=1.15", dict(do_sample=True, temperature=0.8, top_p=0.9, repetition_penalty=1.15)),
]

for name, kwargs in settings:
    generate_once(name, seed=123, **kwargs)


## 5. 直接看下一 token 概率

`generate()` 的本质是循环 forward，然后从最后一个位置的 logits 里选下一个 token。下面只看第一步的候选 token。


In [ ]:
with torch.inference_mode():
    outputs = model(**inputs)

last_logits = outputs.logits[:, -1, :]

for temperature in [0.3, 1.0, 1.5]:
    probs = torch.softmax(last_logits / temperature, dim=-1)
    values, indices = torch.topk(probs[0], k=10)
    print("=" * 80)
    print("temperature =", temperature)
    for rank, (token_id, prob) in enumerate(zip(indices.tolist(), values.tolist()), start=1):
        token = tokenizer.decode([token_id], skip_special_tokens=False)
        print(f"{rank:02d} id={token_id:<8} prob={prob:.4f} token={token!r}")


## 6. 面试总结

- `max_new_tokens` 控制输出上限，`max_length` 包含输入长度，二者不要混淆。
- greedy/beam 更确定，sampling 更开放；面试要能解释为什么同一个 prompt 多次输出不同。
- temperature 调整分布尖锐程度，top-k/top-p 裁剪候选集合。
- repetition penalty 能缓解重复，但不是解决幻觉或事实错误的手段。
- 生产配置要按任务区分：代码/抽取更保守，创作/头脑风暴可以更开放。
- 排障时固定 seed、记录完整 generation config，并保存 prompt token 数和输出 token 数。
